## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [ ]:
import seaborn as sns

In [ ]:
titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)
titanic_data

**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [ ]:
titanic_data.isnull().sum()

### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [ ]:
n_rows = titanic_data.shape[0]
titanic_data = titanic_data.dropna(axis=1, thresh=n_rows // 2)

n_columns = titanic_data.shape[1]
titanic_data = titanic_data.dropna(thresh=n_columns // 2)

titanic_data

### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [ ]:
median_ages = {
    "man": round(titanic_data[titanic_data["who"] == "man"]["age"].median()),
    "woman": round(titanic_data[titanic_data["who"] == "woman"]["age"].median()),
    "child": round(titanic_data[titanic_data["who"] == "child"]["age"].median()),
}

for category, median_age in median_ages.items():
    mask = (titanic_data["who"] == category) & (titanic_data["age"].isna())
    titanic_data.loc[mask, "age"] = median_age

titanic_data

### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [ ]:
titanic_data = titanic_data.dropna()
titanic_data

### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [ ]:
print(titanic_data["embark_town"].value_counts().idxmax())

### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [ ]:
all_passengers = titanic_data.shape[0]
if all_passengers == 0:
    raise ValueError("Division by zero")

count_survived = titanic_data["survived"].sum()

percent_survived = round(count_survived / all_passengers * 100, 2)
percent_survived

### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [ ]:
cnts_surv_passengers = titanic_data[titanic_data["survived"] == 1]["embark_town"].value_counts()
cnts_surv_passengers

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [ ]:
cnt_all_passengers_in_class = titanic_data["class"].value_counts()
if any(cnt_all_passengers_in_class.values == 0):
    raise ValueError("Division by zero")

cnt_surv_passengers_in_class = titanic_data[titanic_data["survived"] == 1]["class"].value_counts()

percent_survived_in_class = round(
    cnt_surv_passengers_in_class / cnt_all_passengers_in_class * 100, 2
)
percent_survived_in_class

### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [ ]:
cnt_all_rich = (titanic_data["fare"] >= 100).sum()
if cnt_all_rich == 0:
    raise ValueError("Division by zero")

cnt_surv_rich = ((titanic_data["survived"] == 1) & (titanic_data["fare"] >= 100)).sum()

percent_surv_rich = round(cnt_surv_rich / cnt_all_rich * 100, 2)
percent_surv_rich

### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [ ]:
((titanic_data["who"] == "child") & titanic_data["alone"]).sum()

Какие выводы вы можете сделать о выживших пассажирах Титаника? 

1) После очистки таблицы от пропущенных данных общий процент выживших составил 38.25%.

2) Ключевым фактором выживания на Титанике был социально-экономический статус.
На выживаемость пассажира сильно влиял его класс. Проценты выживших людей по разным классам: первый класс — 62.62%, второй — 47.28%, третий — всего 24.24%.
Богатые пассажиры (стоимость билета >= 100$) выживали значительно чаще — 73.58% выживших.

3) Наибольшее абсолютное число выживших дал Саутгемптон (217 человек), так в этом порту село большинство пассажиров.